# Sequential Workflow (LLM Integrated)

In [46]:
import os
from langgraph.graph import StateGraph, START, END
from huggingface_hub import InferenceClient
from typing import TypedDict
from dotenv import load_dotenv

In [54]:
load_dotenv()

True

In [55]:
client = InferenceClient(
    api_key=os.environ['HF_TOKEN'],
)


def invoke_model(message, model="meta-llama/Llama-3.2-1B-Instruct:novita"):
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": message}
        ],
    )
    return completion.choices[0].message["content"]

In [56]:
# create a state
class LLMState(TypedDict):

    question: str
    answer: str


In [57]:
def llm_qa(state: LLMState) ->LLMState:

    # extract the question from state
    question = state['question']


    # form a prompt
    prompt = question

    # ask the question to the llm
    answer = invoke_model(prompt)

    # update the answer to the llm

    state['answer'] = answer
    
    return state

In [58]:
# crete our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow= graph.compile()

In [59]:
# Execute 

initial_state = {'question': "What is capital of country?"}
final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'What is capital of country?', 'answer': 'The capital of a country is the city or area that serves as the country\'s government and administrative center. It is often the largest city, the seat of the national government, and the main hub for the country\'s politics, economy, culture, and infrastructure.\n\nExamples of countries with their capitals include:\n\n* United States: Washington, D.C.\n* United Kingdom: London\n* Canada: Ottawa\n* Australia: Canberra\n* Germany: Berlin\n* France: Paris\n* Italy: Rome\n\nThe capital is usually a major urban center and often has a rich history, cultural significance, and economic importance. In some cases, the capital may be a smaller town or city that has grown into a major urban center due to its strategic location or economic opportunities.\n\nThe term "capital" can also refer to the seat of power or authority, used in other contexts such as:\n\n* The capital of the United States is a city called Washington, D.C.\n* The capital o